# 03. 어린이보호구역 전처리

전국 어린이보호구역 표준데이터를 229개 사고지역 기준으로 매칭하고,
지역별 보호구역 수와 CCTV 설치 현황을 집계합니다.

**출력**
- `../data_clean/schoolzone_2026_clean.csv`

In [1]:
import numpy as np
import pandas as pd

schoolzone = pd.read_csv(
    "../data_raw/전국어린이보호구역표준데이터.csv",
    encoding="cp949"
)

accident = pd.read_csv(
    "../data_clean/accident_2020_2025_child_clean.csv",
    encoding="utf-8-sig"
)

print("보호구역 원본:", schoolzone.shape)
display(schoolzone.head())

보호구역 원본: (14642, 16)


,시설종류,대상시설명,소재지도로명주소,소재지지번주소,위도,경도,관리기관명,관할경찰서명,CCTV설치여부,CCTV설치대수,어린이보호구역지정여부,어린이보호구역지정여부수정일자,보호구역도로폭,데이터기준일자,제공기관코드,제공기관명
0,어린이집,큰숲 어린이집,제주특별자치도 제주시 월산남2길 8,제주특별자치도 제주시 노형동 3289-1,33.471106,126.464304,제주특별자치도 자치경찰단 교통정보센터,제주서부경찰서,Y,1.0,NaN,NaN,NaN,2026-07-31,6500000,제주특별자치도
1,초등학교,보물섬학교,제주특별자치도 제주시 아연로 361,제주특별자치도 제주시 오등동 1555-3,33.470336,126.522012,제주특별자치도 자치경찰단 교통정보센터,제주동부경찰서,N,0.0,NaN,NaN,NaN,2026-07-31,6500000,제주특별자치도
2,어린이집,샘어린이집,제주특별자치도 서귀포시 대정읍 에듀시티로 84,제주특별자치도 서귀포시 대정읍 보성리 2410,33.288169,126.281997,제주특별자치도 자치경찰단 교통정보센터,제주서귀포경찰서,N,0.0,NaN,NaN,NaN,2026-07-31,6500000,제주특별자치도
3,어린이집,꿈동산어린이집,제주특별자치도 서귀포시 남원읍 태위로20번길 22,제주특별자치도 서귀포시 남원읍 위미리 4183-9,33.270383,126.649961,제주특별자치도 자치경찰단 교통정보센터,제주서귀포경찰서,N,0.0,NaN,NaN,NaN,2026-07-31,6500000,제주특별자치도
4,어린이집,공립중흥S-클래스어린이집,제주특별자치도 서귀포시 대청로 12,제주특별자치도 서귀포시 강정동 195,33.252324,126.500488,제주특별자치도 자치경찰단 교통정보센터,제주서귀포경찰서,Y,1.0,NaN,NaN,NaN,2026-07-31,6500000,제주특별자치도


## 1) 기본 점검 및 대표 주소 생성

In [2]:
print("중복 행:", schoolzone.duplicated().sum())
print("\nCCTV 설치여부")
print(schoolzone["CCTV설치여부"].value_counts(dropna=False))

print("\nCCTV 설치대수 결측치:",
      schoolzone["CCTV설치대수"].isna().sum())

schoolzone["주소"] = (
    schoolzone["소재지도로명주소"]
    .fillna(schoolzone["소재지지번주소"])
)

print("주소 결측:", schoolzone["주소"].isna().sum())

중복 행: 0

CCTV 설치여부
CCTV설치여부
Y    11818
N     2824
Name: count, dtype: int64

CCTV 설치대수 결측치: 2689
주소 결측: 0


## 2) 주소 행정구역 정규화 및 229개 지역 매칭

In [3]:
accident_regions = sorted(
    accident["sido_sgg_nm"].dropna().unique(),
    key=len,
    reverse=True
)

gwangju_gu = {"동구", "서구", "남구", "북구", "광산구"}

def normalize_address(address):
    if pd.isna(address):
        return address

    parts = str(address).split()

    # 데이터에 나타나는 통합 행정명칭을 기존 분석기준으로 환원
    if len(parts) >= 2 and parts[0] == "전남광주통합특별시":
        if parts[1] in gwangju_gu:
            parts[0] = "광주광역시"
        else:
            parts[0] = "전라남도"

    address = " ".join(parts)

    return (
        address
        .replace("강원특별자치도", "강원도", 1)
        .replace("전북특별자치도", "전라북도", 1)
        .replace("제주특별자치도", "제주도", 1)
        .replace("인천광역시 서해구", "인천광역시 서구", 1)
        .replace("인천광역시 검단구", "인천광역시 서구", 1)
        .replace("경북 ", "경상북도 ", 1)
    )

def match_region(address):
    if pd.isna(address):
        return None

    for region in accident_regions:
        if address.startswith(region):
            return region

    return None

schoolzone["주소_정규화"] = schoolzone["주소"].apply(normalize_address)
schoolzone["sido_sgg_nm"] = schoolzone["주소_정규화"].apply(match_region)

print("1차 매칭 성공:", schoolzone["sido_sgg_nm"].notna().sum())
print("1차 매칭 실패:", schoolzone["sido_sgg_nm"].isna().sum())

1차 매칭 성공: 14637
1차 매칭 실패: 5


## 3) 소수 미매칭 행 수동 보정

아래 인덱스는 기존 전처리에서 주소·관리기관·좌표를 직접 확인한 뒤 확정한 항목입니다.
원본 파일이 변경되면 반드시 다시 검토해야 합니다.

In [4]:
manual_match = {
    5319: "광주광역시 북구",
    12506: "서울특별시 도봉구",
    13240: "강원도 삼척시",
    13615: "서울특별시 도봉구",
    13950: "경기도 용인시",
}

for idx, region in manual_match.items():
    if idx in schoolzone.index:
        schoolzone.loc[idx, "sido_sgg_nm"] = region

print("수동 보정 후 매칭 실패:",
      schoolzone["sido_sgg_nm"].isna().sum())

수동 보정 후 매칭 실패: 0


## 4) 중복 시설 정리

In [5]:
schoolzone["데이터기준일자"] = pd.to_datetime(
    schoolzone["데이터기준일자"],
    errors="coerce"
)

schoolzone = schoolzone.sort_values(
    "데이터기준일자",
    ascending=False
)

schoolzone_clean = schoolzone.drop_duplicates(
    subset=["대상시설명", "주소"],
    keep="first"
).copy()

print("중복 제거 전:", len(schoolzone))
print("중복 제거 후:", len(schoolzone_clean))
print("지역 매칭 실패:", schoolzone_clean["sido_sgg_nm"].isna().sum())

중복 제거 전: 14642
중복 제거 후: 14192
지역 매칭 실패: 0


## 5) 지역별 보호구역·CCTV 집계

In [6]:
temp = schoolzone_clean.copy()

temp["CCTV설치대수"] = pd.to_numeric(
    temp["CCTV설치대수"],
    errors="coerce"
)

# CCTV 미설치(N)는 설치대수 0
temp.loc[
    temp["CCTV설치여부"] == "N",
    "CCTV설치대수"
] = 0

schoolzone_summary = (
    temp.groupby("sido_sgg_nm")
    .agg(
        schoolzone_count=("대상시설명", "size"),
        cctv_installed_zone_count=(
            "CCTV설치여부",
            lambda x: (x == "Y").sum()
        )
    )
    .reset_index()
)

cctv_total = (
    temp.groupby("sido_sgg_nm")["CCTV설치대수"]
    .sum(min_count=1)
    .reset_index(name="cctv_count_total")
)

schoolzone_summary = schoolzone_summary.merge(
    cctv_total,
    on="sido_sgg_nm",
    how="left"
)

# CCTV는 설치됐지만 대수가 빠진 지역은 총대수를 알 수 없음
unknown_regions = temp.loc[
    (temp["CCTV설치여부"] == "Y") &
    (temp["CCTV설치대수"].isna()),
    "sido_sgg_nm"
].unique()

schoolzone_summary.loc[
    schoolzone_summary["sido_sgg_nm"].isin(unknown_regions),
    "cctv_count_total"
] = np.nan

## 6) 보호구역 원본 미제공 지역 표시 및 저장

In [7]:
accident_regions_set = set(accident["sido_sgg_nm"].dropna().unique())
schoolzone_regions = set(schoolzone_summary["sido_sgg_nm"].dropna().unique())

missing_regions = sorted(accident_regions_set - schoolzone_regions)
print("보호구역 원본 미제공 지역:", missing_regions)

missing_rows = pd.DataFrame({
    "sido_sgg_nm": missing_regions,
    "schoolzone_count": np.nan,
    "cctv_installed_zone_count": np.nan,
    "cctv_count_total": np.nan,
})

schoolzone_final = pd.concat(
    [schoolzone_summary, missing_rows],
    ignore_index=True
).drop_duplicates(subset=["sido_sgg_nm"], keep="first")

schoolzone_final["schoolzone_data_available"] = (
    schoolzone_final["schoolzone_count"].notna()
).astype(int)

schoolzone_final = (
    schoolzone_final
    .sort_values("sido_sgg_nm")
    .reset_index(drop=True)
)

print("최종 지역 수:", len(schoolzone_final))
print(schoolzone_final.isna().sum())

보호구역 원본 미제공 지역: ['전라북도 김제시', '충청북도 음성군']
최종 지역 수: 229
sido_sgg_nm                   0
schoolzone_count              2
cctv_installed_zone_count     2
cctv_count_total             42
schoolzone_data_available     0
dtype: int64


In [8]:
schoolzone_final.to_csv(
    "../data_clean/schoolzone_2026_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", schoolzone_final.shape)
display(schoolzone_final.head())

저장 완료: (229, 5)


,sido_sgg_nm,schoolzone_count,cctv_installed_zone_count,cctv_count_total,schoolzone_data_available
0,강원도 강릉시,71.0,63.0,213.0,1
1,강원도 고성군,10.0,9.0,40.0,1
2,강원도 동해시,29.0,20.0,74.0,1
3,강원도 삼척시,18.0,18.0,75.0,1
4,강원도 속초시,14.0,14.0,60.0,1
